# Module 4.3 — Functional Toolkit

### Python Mastery · Track 4: Functional Programming & Metaprogramming

The standard library provides a rich set of functional tools. `map`, `filter`, and `reduce` express common transformations; `functools` offers utilities for working with functions; and `itertools` builds efficient iterators for combinations, grouping, and infinite sequences. This module surveys the most useful of each.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Compare each tool with the comprehension or loop it replaces.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Use `map` and `filter`, and know when a comprehension is clearer.
2. Use `functools.reduce` to fold a sequence to a single value.
3. Apply `functools.partial`, `lru_cache`, and `singledispatch`.
4. Use core `itertools` tools: `chain`, `cycle`, `count`, `groupby`, `combinations`, `product`.
5. Combine these tools to process data efficiently.

**Prerequisites:** Tracks 1 to 3, and Modules 4.1 to 4.2.

---

## Part 1 · `map` and `filter`

`map(func, iterable)` applies a function to every item, and `filter(func, iterable)` keeps items for which a function returns true. Both return lazy iterators, so wrap them in `list()` to see all results. In modern Python a comprehension is often as clear or clearer, but `map`/`filter` shine when you already have a named function to apply.

In [ ]:
numbers = [1, 2, 3, 4, 5]

# map applies a function to each item; the result is a lazy iterator.
squared = map(lambda n: n * n, numbers)
print("squared:", list(squared))

# filter keeps items where the predicate is true.
evens = filter(lambda n: n % 2 == 0, numbers)
print("evens:", list(evens))

# With a named function, map reads very cleanly.
words = ["hello", "world"]
print("upper:", list(map(str.upper, words)))

# The comprehension equivalents, often preferred for readability:
print("comprehension:", [n * n for n in numbers])
print("filter comprehension:", [n for n in numbers if n % 2 == 0])

## Part 2 · `functools.reduce`

`reduce(func, iterable)` folds a sequence into a single value by repeatedly applying a two-argument function: it combines the first two items, then combines that result with the next, and so on. Many reductions have clearer built-ins (`sum`, `max`, `min`), so reach for `reduce` when no built-in fits.

In [ ]:
from functools import reduce

numbers = [1, 2, 3, 4, 5]

# Multiply everything together (there is no built-in 'product').
product = reduce(lambda acc, n: acc * n, numbers)
print("product:", product)               # 120

# With an explicit starting value (useful for an empty sequence).
total = reduce(lambda acc, n: acc + n, numbers, 100)
print("sum starting from 100:", total)    # 115

# Finding a maximum manually, to show the folding mechanism (use max() in practice).
largest = reduce(lambda a, b: a if a > b else b, numbers)
print("largest:", largest)

## Part 3 · `functools`: `partial`, `lru_cache`, `singledispatch`

`functools` provides several practical utilities:

- `partial(func, ...)` fixes some arguments of a function, producing a simpler one.
- `lru_cache` caches results of expensive calls automatically.
- `singledispatch` enables a function to behave differently based on the type of its first argument, a clean alternative to type-checking branches.

In [ ]:
from functools import partial

def power(base, exponent):
    return base ** exponent

# Fix the exponent to create specialised functions.
square = partial(power, exponent=2)
cube = partial(power, exponent=3)
print("square(5):", square(5))
print("cube(2):", cube(2))

# partial is handy for configuring callbacks and keys.
from functools import partial
def greet(greeting, name):
    return f"{greeting}, {name}"
say_hi = partial(greet, "Hi")
print(say_hi("Asha"))

In [ ]:
from functools import lru_cache

call_count = []

@lru_cache(maxsize=None)
def expensive(n):
    call_count.append(n)         # record genuine computations
    return n * n

print(expensive(4), expensive(4), expensive(5))
print("computed only for:", call_count)     # [4, 5] -- repeats are cached

In [ ]:
from functools import singledispatch

@singledispatch
def describe(value):
    return f"some value: {value}"

@describe.register
def _(value: int):
    return f"an integer: {value}"

@describe.register
def _(value: list):
    return f"a list of {len(value)} items"

# The implementation is chosen by the type of the first argument.
print(describe(42))
print(describe([1, 2, 3]))
print(describe("hello"))         # falls back to the default

## Part 4 · `itertools`: Building Efficient Iterators

`itertools` provides iterators that are memory-efficient and composable.

- `chain` links several iterables into one sequence.
- `count`, `cycle`, and `repeat` are infinite generators (take a slice with `islice`).
- `combinations` and `permutations` produce groupings.
- `product` gives the Cartesian product, like nested loops.

In [ ]:
from itertools import chain, count, cycle, repeat, islice

# chain: iterate over several sequences as if they were one.
print("chained:", list(chain([1, 2], (3, 4), "ab")))

# count: an endless counter; islice takes a finite number of values.
print("count from 10, step 2, first 5:", list(islice(count(10, 2), 5)))

# cycle: repeat a sequence forever; take the first 7.
print("cycle:", list(islice(cycle(["a", "b", "c"]), 7)))

# repeat: the same value a fixed number of times.
print("repeat:", list(repeat("x", 4)))

In [ ]:
from itertools import combinations, permutations, product

items = ["A", "B", "C"]

# combinations: unordered selections of a given size.
print("pairs (combinations):", list(combinations(items, 2)))

# permutations: ordered arrangements.
print("ordered pairs (permutations):", list(permutations(items, 2)))

# product: all combinations across iterables, like nested for-loops.
print("product of sizes and colors:")
for size, color in product(["S", "L"], ["red", "blue"]):
    print(" ", size, color)

`groupby` groups **consecutive** items that share a key. Because it only groups runs that are adjacent, the data must usually be sorted by the same key first.

In [ ]:
from itertools import groupby

people = [
    {"city": "Pune", "name": "Asha"},
    {"city": "Pune", "name": "Chen"},
    {"city": "Delhi", "name": "Ben"},
    {"city": "Delhi", "name": "Dev"},
]

# Sort by the grouping key first, then group consecutive runs.
people.sort(key=lambda p: p["city"])
for city, group in groupby(people, key=lambda p: p["city"]):
    names = [person["name"] for person in group]
    print(f"{city}: {names}")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — map and filter together (Easy)

In [ ]:
numbers = [1, 2, 3, 4, 5, 6]
# Square the numbers, then keep only those above 10.
squared = map(lambda n: n * n, numbers)
big = filter(lambda n: n > 10, squared)
print(list(big))      # [16, 25, 36]

### Example 2 — partial for configuration (Easy)

In [ ]:
from functools import partial

def multiply(a, b):
    return a * b

triple = partial(multiply, 3)
print("triple(10):", triple(10))
print("triple(4):", triple(4))

### Example 3 — reduce for a running computation (Medium)

In [ ]:
from functools import reduce

transactions = [100, -30, 50, -20, 10]
# Fold the transactions into a final balance, starting from 0.
balance = reduce(lambda acc, amount: acc + amount, transactions, 0)
print("final balance:", balance)

# Build a sentence by joining with reduce (str.join is better in practice; this shows folding).
words = ["functional", "programming", "is", "elegant"]
sentence = reduce(lambda a, b: a + " " + b, words)
print(sentence)

### Example 4 — singledispatch for type-based behaviour (Medium)

In [ ]:
from functools import singledispatch

@singledispatch
def to_text(value):
    return str(value)

@to_text.register
def _(value: list):
    return ", ".join(str(v) for v in value)

@to_text.register
def _(value: dict):
    return "; ".join(f"{k}={v}" for k, v in value.items())

print(to_text(42))
print(to_text([1, 2, 3]))
print(to_text({"a": 1, "b": 2}))

### Example 5 — Combinatorics with itertools (Difficult)

In [ ]:
from itertools import combinations, product

# All ways to choose 2 toppings from 4 (order does not matter).
toppings = ["cheese", "mushroom", "olive", "pepper"]
print("topping pairs:")
for pair in combinations(toppings, 2):
    print(" ", pair)

# Every possible small meal: one size and one drink.
print("meals:")
for size, drink in product(["small", "large"], ["tea", "coffee"]):
    print(f"  {size} with {drink}")

### Example 6 — Grouping and summarising a dataset (Difficult)

In [ ]:
from itertools import groupby

sales = [
    {"region": "North", "amount": 100},
    {"region": "South", "amount": 200},
    {"region": "North", "amount": 150},
    {"region": "South", "amount": 50},
    {"region": "East", "amount": 300},
]

# Sort by region, group, and total each group.
sales.sort(key=lambda s: s["region"])
summary = {}
for region, group in groupby(sales, key=lambda s: s["region"]):
    summary[region] = sum(item["amount"] for item in group)

print("totals by region:", summary)
print("best region:", max(summary, key=summary.get))

---

## Exercises

**Exercise 1 (Easy).** Use `map` to convert the list `["1", "2", "3"]` of strings into a list of integers, then print the result.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use `filter` to keep only the words longer than three letters from the list below.

In [ ]:
words = ["cat", "tiger", "ox", "leopard", "ant"]
# Your solution here


**Exercise 3 (Medium).** Use `functools.reduce` to find the product of all numbers in `[2, 3, 4, 5]`.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use `functools.partial` to create a function `to_base_2` from `int(x, base)` that converts a binary string to an integer, then convert `"1010"`.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Given the list of records below, sort and group by `team` using `itertools.groupby`, and print each team with the list of its members' names.

In [ ]:
records = [
    {"team": "B", "name": "Asha"},
    {"team": "A", "name": "Ben"},
    {"team": "B", "name": "Chen"},
    {"team": "A", "name": "Dev"},
]
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
print(list(map(int, ["1", "2", "3"])))

**Solution 2**

In [ ]:
words = ["cat", "tiger", "ox", "leopard", "ant"]
print(list(filter(lambda w: len(w) > 3, words)))

**Solution 3**

In [ ]:
from functools import reduce
print(reduce(lambda a, b: a * b, [2, 3, 4, 5]))

**Solution 4**

In [ ]:
from functools import partial
to_base_2 = partial(int, base=2)
print(to_base_2("1010"))    # 10

**Solution 5**

In [ ]:
from itertools import groupby
records = [
    {"team": "B", "name": "Asha"},
    {"team": "A", "name": "Ben"},
    {"team": "B", "name": "Chen"},
    {"team": "A", "name": "Dev"},
]
records.sort(key=lambda r: r["team"])
for team, group in groupby(records, key=lambda r: r["team"]):
    print(team, [r["name"] for r in group])

---

## Summary and Key Points

- `map` applies a function to every item and `filter` keeps items passing a predicate; both are lazy, and a comprehension is often clearer unless a named function applies.
- `functools.reduce` folds a sequence to one value with a two-argument function and an optional start; prefer built-ins like `sum`, `max`, and `min` when they fit.
- `functools.partial` fixes arguments to specialise a function; `lru_cache` caches results automatically; `singledispatch` selects an implementation by the first argument's type.
- `itertools` builds efficient iterators: `chain` joins sequences; `count`, `cycle`, `repeat` are infinite (slice with `islice`); `combinations`, `permutations`, and `product` cover groupings.
- `groupby` groups only consecutive items by key, so sort by the same key first.

### Next module: 4.4 — Metaclasses & Class Customisation

The next module looks at how classes themselves are created, using `type` as a class factory, custom metaclasses, and the hooks `__init_subclass__` and `__set_name__`.